# Portfolio dashboard

Run the cells from top to bottom. Only the last cell is used day to day.

1. **First time only:** install the packages (first code cell), then restart the kernel.
2. **Once per session:** run the setup cell. It defines everything and downloads nothing.
3. **After a purchase or sale:** download the bank's History PDF into Downloads (any name starting with "History") and run the last cell. It adds the new transaction to `transactions-YYYY-MM-DD.csv` (your permanent record), finds the stock's Yahoo symbol from the name in the PDF, works out what you hold and draws the dashboard.
4. **Any other day:** just run the last cell. If a portfolio Excel in Downloads is at least as recent as your last transaction, it is used for the exact cash balance; otherwise cash is estimated from your last cash file.
5. **Dividend tax:** `dividend-tax.csv` (created next to the notebook) lists the tax withheld at source for each market. Edit it freely; set a row's status to `manual` and the code never changes it. When a History PDF contains a dividend from a market whose rate differs from the table, the code shows the real rate and asks before updating the file.


## First time? Start here

**What this does:** it reads your bank's exports, works out what you hold and what you really paid, fetches live prices from Yahoo Finance and draws a full dashboard: allocation, gains, returns over time, a comparison with IWDA, correlation and risk.

### 1. What you need
- Python 3.9 or newer and a Jupyter notebook (PyCharm, JupyterLab or VS Code all work).
- An internet connection (prices and dividend history come from Yahoo).
- Your bank's exports (below). The code was written for Keytrade Belgium's French-language exports.

### Try it first with the demo data
The `demo/` folder holds fictional History PDFs and a portfolio Excel in the bank's format. In the setup cell, set `DOWNLOADS = "demo"`, then run the last cell. No real account is needed (live prices still come from Yahoo).

### 2. Get your files from the banking app
Put them all in your **Downloads** folder (or the folder named in the `DOWNLOADS` setting). File names don't matter for the Excel; the PDFs must start with `History`.
1. **History PDFs:** export your transaction history as a PDF, once for each stock you have ever bought. This is the permanent record of what you bought, when, at what price and with what fees.
2. **Portfolio Excel** (recommended, once): export your portfolio as Excel (3+ sheets: positions, orders, cash). It gives your exact cash balance. It is optional after the first run.

### 3. Run the notebook
1. Run the **install** cell, then **restart the kernel** (first time only).
2. Run the **setup** cell. It only defines things.
3. Run the **last** cell. The first time it will:
   - read your PDFs and create `transactions-DATE.csv` (buys and sells only, with the Yahoo symbol of each stock, found from its name; if it can't, it asks you to type the symbol, for example `AI.PA`);
   - create `holdings-DATE.csv`, `cash-DATE.csv` and `dividend-tax.csv`, then draw the dashboard.
   - If a file with the same name already exists with different content, it asks before overwriting. Answer `y` or `n`.

All these files appear **next to the notebook**. They are your data: keep them (back them up or put the folder in iCloud/Dropbox). If you move to another computer, copy them with the notebook.

### 4. Every day after
- **Nothing changed:** run the last cell.
- **You bought or sold something:** download that stock's new History PDF into Downloads and run the last cell. Only the new transactions are added.
- **You want the exact cash balance:** export the Excel again before running.

### 5. Check these settings once
- **`BELGIAN_TAX`** in the setup cell: 30% is the Belgian withholding on dividends. Change it if you are not a Belgian resident.
- **`dividend-tax.csv`:** the tax withheld at source per country (US 30%, France 25%, Netherlands 15% are filled in; others are marked "to check"). The code corrects it from the dividend lines of your PDFs and asks before changing the file.
- **Currencies:** EUR and USD are supported. Anything else stops with a message.
- **Markets in the Excel:** Paris, Amsterdam and US are known. For another market, add its name to `MARKET_SUFFIX` in the setup cell (the error message tells you the name).

### 6. If something stops
| Message | Meaning |
|---|---|
| *No transactions yet and no History PDF found* | No PDF starting with `History` in Downloads. |
| *No portfolio export found* | No Excel with the bank's layout in Downloads or the notebook folder. Only needed if there is no transactions file. |
| *Unknown market(s) in column B* | Add the market to `MARKET_SUFFIX`. |
| *Not matched with the ledger* | The shares in your Excel differ from your transactions: a PDF is missing. Download it and run again. |
| *Market not in the tax table* | Add a row to `dividend-tax.csv`; 30% is assumed meanwhile. |
| *No exchange rate set up for …* | A currency other than EUR or USD. |
| *contains sales … which the cost calculation does not handle* | Partial sales are not supported yet. |

### 7. Privacy
The notebook itself contains none of your data, but once you run it, it saves your tables and charts inside. **Clear all outputs** before sending it to someone, and never send the CSV, Excel or PDF files.


## 1. Install (first time only)

In [ ]:
# First time only (then restart the kernel). Safe to run again.
%pip install pandas numpy yfinance plotly nbformat ipython openpyxl pypdf

## 2. Setup (once per session)

In [ ]:
# ===== SETUP: run this cell once per session. It only defines things; nothing is downloaded or drawn. =====
import os
import glob
import re
import datetime as dt
from functools import lru_cache

import warnings
warnings.filterwarnings("ignore", message="urllib3 v2 only supports OpenSSL")
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")     # Excel extensions openpyxl skips: harmless

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

pio.templates["clear"] = go.layout.Template(layout=dict(paper_bgcolor="rgba(0,0,0,0)"))
pio.templates.default = "plotly_white+clear"

pd.options.display.float_format = "{:,.2f}".format

# ---------- settings ----------
DOWNLOADS = "~/Downloads"      # where your bank's History PDFs and portfolio Excel are. For the demo: DOWNLOADS = "demo"

# ---------- price download (each series is downloaded once, then reused by every chart) ----------
@lru_cache(maxsize=None)
def _download_close(symbol, period, interval):
    return yf.Ticker(symbol).history(period=period, interval=interval, auto_adjust=False)["Close"]

def cached_close(symbol, period, interval="1d"):
    return _download_close(symbol, period, interval).copy()      # a copy, so charts can reshape it freely

# ---------- the chart functions, as built so far ----------
# --- asset / sector allocation bars ---
tracker_palette = ["#049faf", "#f04923", "#3c2415", "#dbcbb2", "#253e88"]
tint_levels = [1.0, 0.6, 0.35]

# ---------- colours: one colour = one meaning, the same in every chart ----------
ASSET_COLORS = {"ETFs": "#4e576a", "Equities": "#bfc6d4", "Cash": "#0055A4"}   # slate / light blue-grey / capital blue

SECTOR_COLORS = {                      # each sector keeps its colour, whatever its rank
    "Technology": "#88ccee", "Financial Services": "#cc6677", "Industrials": "#ddcc77",
    "Healthcare": "#117733", "Consumer Cyclical": "#332288", "Communication Services": "#aa4499",
    "Consumer Defensive": "#6699cc", "Energy": "#999933", "Basic Materials": "#882255",
    "Utilities": "#661100", "Real Estate": "#888888", "Unknown": "#bbbbbb",
}

HOLDING_COLORS_FIXED = {               # each holding keeps its colour in every chart that names it
    "IWDA.AS": "#4e576a",              # the ETF: same slate as the ETFs class
}
STOCK_TINT_BASES = ["#049faf", "#ffbc05", "#fb8b1e", "#253e88", "#009E4D"]
_holding_colors = dict(HOLDING_COLORS_FIXED)

def tint(hex_color, strength):
    """Lighter version of a colour (strength 1 = the colour itself, 0 = white), still a solid hex."""
    h = hex_color.lstrip("#")
    r, g, b = (int(h[k:k + 2], 16) for k in (0, 2, 4))
    mix = lambda c: round(255 - (255 - c) * strength)
    return "#%02x%02x%02x" % (mix(r), mix(g), mix(b))

def assign_holding_colors(holdings):
    """New holdings get a lighter version of the existing colours (slate for ETFs); known ones never change."""
    etfs = stocks = 0
    for symbol, kind in zip(holdings["symbol"], holdings["asset_type"]):
        if symbol in _holding_colors:
            continue
        if kind == "ETFs":
            etfs += 1
            _holding_colors[symbol] = tint("#4e576a", max(0.3, 1 - 0.3 * etfs))
        else:
            base = STOCK_TINT_BASES[stocks % len(STOCK_TINT_BASES)]
            level = [1.0, 0.6, 0.35][min(stocks // len(STOCK_TINT_BASES), 2)]
            _holding_colors[symbol] = tint(base, level)
            stocks += 1

def holding_color(symbol):
    return _holding_colors.get(symbol, "#999999")

def allocation_bar(values, title, palette=tracker_palette, colors=None):
    values = values.sort_values(ascending=False)
    total = values.sum()
    fig = go.Figure()
    for i, (name, value) in enumerate(values.items()):
        fixed = colors is not None and name in colors            # colour chosen by name, not by rank
        fig.add_trace(go.Bar(
            x=[value],
            y=[title],
            orientation="h",
            name=f"{name}: €{value:,.0f} ({value / total:.1%})",
            marker_color=colors[name] if fixed else palette[i % len(palette)],
            marker_line_color="white",
            marker_line_width=1,
            opacity=1.0 if fixed else tint_levels[min(i // len(palette), len(tint_levels) - 1)],
            hovertemplate=f"<b>{name}</b><br>€{value:,.0f}<br>{value / total:.1%}<extra></extra>",
        ))

    legend_item_height = 24
    top, bar_height = 50, 80
    bottom = legend_item_height * len(values) + 10

    fig.update_layout(
        barmode="stack",
        title=title,
        height=top + bar_height + bottom,
        margin=dict(t=top, b=bottom, l=20, r=20),
        xaxis_visible=False,
        yaxis_visible=False,
        legend=dict(x=0, y=-0.2, yanchor="top", traceorder="normal", font=dict(size=12), maxheight=1),
    )
    fig.update_layout(clickmode="event+select")
    fig.update_traces(selected=dict(marker=dict(opacity=1)), unselected=dict(marker=dict(opacity=0.25)))
    return fig

# --- holdings doughnut ---
def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = (int(h[k:k + 2], 16) for k in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

def doughnut(values, title, palette=tracker_palette, fixed_colors=None):
    values = values.sort_values(ascending=False)
    total = values.sum()
    fixed_colors = fixed_colors or {}
    colors, rank = [], 0
    for name in values.index:
        if name in fixed_colors:                          # e.g. Cash keeps its own colour
            colors.append(hex_to_rgba(fixed_colors[name], 1.0))
        else:                                             # the holdings are coloured by rank, as before
            colors.append(hex_to_rgba(palette[rank % len(palette)], tint_levels[min(rank // len(palette), len(tint_levels) - 1)]))
            rank += 1
    fig = go.Figure(go.Pie(
        labels=values.index,
        values=values.values,
        hole=0.55,
        sort=False,
        direction="clockwise",
        marker=dict(colors=colors, line=dict(color="white", width=2)),
        texttemplate="%{percent:.1%}",
        hovertemplate="<b>%{label}</b><br>€%{value:,.0f}<br>%{percent:.1%}<extra></extra>",
    ))
    fig.update_layout(
        title=title,
        height=420,
        margin=dict(t=60, b=20, l=20, r=20),
        legend=dict(font=dict(size=12)),
        annotations=[dict(text=f"<b>€{total:,.0f}</b><br>Total", x=0.5, y=0.5, showarrow=False, font=dict(size=16))],
    )
    return fig

# --- gain / loss and cost-to-value ---
GREEN_PALETTE = ["#c2f2e2", "#97e1af", "#81c17b", "#5c9f32", "#52811b"]  # light -> dark
RED_PALETTE = ["#af5852", "#af423b", "#963a34", "#89221b", "#7c0a02"]    # light -> dark
BASE_BLUE = "#0055A4"       # blue of the French flag
DIVIDEND_COLOR = "#212631"  # dark blue-grey, used for dividends only

def shade(pct, full_scale_pct=100):
    palette = GREEN_PALETTE if pct >= 0 else RED_PALETTE
    t = 0.1 + 0.9 * min(abs(pct) / full_scale_pct, 1)
    return sample_colorscale(palette, t)[0]

def signed_eur(x):
    return f"{'+' if x >= 0 else '-'}€{abs(x):,.2f}"

def signed_pct(x):
    return f"{x:+.2f}%"

def gain_loss_chart(holdings, title="Gain / Loss per Position"):
    data = holdings.sort_values("gain_eur", ascending=False)
    gains = data["gain_eur"]
    fig = go.Figure(go.Bar(
        x=data["symbol"],
        y=gains,
        marker_color=[shade(p) for p in data["gain_pct"]],
        text=[signed_eur(g) for g in gains],
        textposition="outside",
        customdata=[signed_pct(p) for p in data["gain_pct"]],
        hovertemplate="<b>%{x}</b><br>%{text}<br>%{customdata}<extra></extra>",
    ))
    fig.update_layout(
        title=title,
        height=400,
        margin=dict(t=60, b=40, l=70, r=20),
        xaxis=dict(showline=False),
        yaxis=dict(tickprefix="€", tickformat=",.0f", zeroline=False, gridcolor="#e6e6e6"),
        plot_bgcolor="white",
    )
    return fig

def cost_to_value_chart(holdings, title="From Cost to Value", base_color=BASE_BLUE):
    data = holdings.sort_values("market_value_eur", ascending=False)
    value, cost = data["market_value_eur"], data["cost_basis_eur"]
    capital = pd.concat([value, cost], axis=1).min(axis=1)
    gain = (value - cost).clip(lower=0)
    loss = (cost - value).clip(lower=0)
    pct = data["gain_pct"]
    pct_labels = [signed_pct(p) for p in pct]
    separator = dict(color="white", width=1.5)

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=data["symbol"], y=capital, name="Capital kept",
        marker=dict(color=base_color, line=separator),
        hovertemplate="<b>%{x}</b><br>Capital kept: €%{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        x=data["symbol"], y=gain, name="Gain", customdata=pct_labels,
        marker=dict(color=[shade(p) for p in pct], line=separator),
        hovertemplate="<b>%{x}</b><br>Gain: +€%{y:,.2f}<br>%{customdata}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        x=data["symbol"], y=loss, name="Loss", customdata=pct_labels,
        marker=dict(color=[shade(p) for p in pct], line=separator),
        hovertemplate="<b>%{x}</b><br>Loss: -€%{y:,.2f}<br>%{customdata}<extra></extra>",
    ))
    has_dividends = data["dividends_eur"].sum() > 0              # no dividend data: no grey segment
    if has_dividends:
        fig.add_trace(go.Bar(
            x=data["symbol"], y=data["dividends_eur"], name="Dividends",
            marker=dict(color=DIVIDEND_COLOR, line=separator),
            hovertemplate="<b>%{x}</b><br>Dividends: €%{y:,.0f}<extra></extra>",
        ))
    legend = "Blue: capital kept  ·  Green: gain  ·  Red: loss" + ("  ·  Grey: dividends" if has_dividends else "")
    fig.update_layout(
        barmode="stack",
        title=dict(text=f"{title}<br><sup>{legend}</sup>"),
        height=420,
        margin=dict(t=90, b=40, l=70, r=20),
        showlegend=False,
        xaxis=dict(showline=False),
        yaxis=dict(tickprefix="€", tickformat=",.0f", gridcolor="#e6e6e6"),
        plot_bgcolor="white",
    )
    return fig

# --- currency exposure ---
CURRENCY_COLORS = {"EUR": "#0068ff", "USD": "#5c9f32"}   # Bloomberg blue for EUR, green for USD

def currency_exposure(holdings, cash=None):
    """Market value in EUR grouped by the currency each position is listed in."""
    by_ccy = holdings.groupby("currency")["market_value_eur"].sum()
    if cash is not None:                                  # optional: cash table with 'currency' and 'amount'
        by_ccy = by_ccy.add(cash.groupby("currency")["amount"].sum(), fill_value=0)
    return by_ccy[by_ccy > 0].sort_values(ascending=False)

def currency_bar(exposure, title="Currency Exposure"):
    total = exposure.sum()
    fig = go.Figure()
    for ccy, value in exposure.items():
        pct = value / total * 100
        fig.add_trace(go.Bar(
            y=[""], x=[value], orientation="h", name=ccy,
            marker=dict(color=CURRENCY_COLORS.get(ccy, "#4e576a"), line=dict(color="white", width=1.5)),
            text=[f"{ccy}  {pct:.1f}%"], textposition="inside", insidetextanchor="middle",
            textfont=dict(color="white", size=14),
            customdata=[[f"€{value:,.0f}", f"{pct:.1f}%"]],
            hovertemplate=f"<b>{ccy}</b><br>%{{customdata[0]}}<br>%{{customdata[1]}}<extra></extra>",
        ))
    fig.update_layout(
        barmode="stack", title=title, height=170, showlegend=False,
        margin=dict(t=60, b=20, l=20, r=20),
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        plot_bgcolor="white",
    )
    return fig

# --- return vs cost (period buttons, zoom, hourly) ---
def close_history(symbol, period, interval="1d"):
    s = cached_close(symbol, period, interval)
    if interval == "1d":
        s.index = s.index.tz_localize(None).normalize()          # plain dates
    else:
        s.index = s.index.tz_convert("Europe/Paris").tz_localize(None)   # one clock for every exchange
    return s[~s.index.duplicated(keep="last")]

def aligned(symbols, period, interval):
    prices = pd.DataFrame({s: close_history(s, period, interval) for s in symbols})
    eurusd = close_history("EURUSD=X", period, interval)
    times = prices.index.union(eurusd.index).sort_values()
    prices = prices.reindex(times).ffill()                       # closed market keeps its last price
    eurusd = eurusd.reindex(times).ffill()
    complete = prices.notna().all(axis=1) & eurusd.notna()
    return prices[complete], eurusd[complete]

def load_history(symbols, daily_period="5y", hourly_period="730d"):
    """Daily prices for the long past, hourly prices for the last ~2 years (Yahoo's hourly limit)."""
    p_d, fx_d = aligned(symbols, daily_period, "1d")
    p_h, fx_h = aligned(symbols, hourly_period, "1h")
    old = p_d.index < p_h.index[0].normalize()
    return pd.concat([p_d[old], p_h]), pd.concat([fx_d[old], fx_h])

def holding_returns(holdings, prices, eurusd):
    returns = {}
    for _, row in holdings.iterrows():
        if row["currency"] == "EUR":
            fx = 1.0
        elif row["currency"] == "USD":
            fx = 1 / eurusd
        else:
            raise ValueError(f"No exchange rate set up for {row['currency']} ({row['symbol']})")
        lots = LEDGER.get("lots")
        mine = lots[lots["symbol"] == row["symbol"]] if lots is not None else None
        if mine is not None and len(mine):                       # real lots: shares and cost build up on each purchase date
            shares_t, cost_t = pd.Series(0.0, index=prices.index), pd.Series(0.0, index=prices.index)
            for lot in mine.itertuples():
                later = prices.index >= lot.date
                shares_t[later] += lot.quantity
                cost_t[later] += lot.cost_eur
            value = shares_t * prices[row["symbol"]] * fx
            returns[row["symbol"]] = (value / cost_t.where(cost_t > 0) - 1) * 100       # empty before the first purchase
        else:
            value = row["shares"] * prices[row["symbol"]] * fx
            returns[row["symbol"]] = (value / row["cost_basis_eur"] - 1) * 100
    return pd.DataFrame(returns)

def rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = (int(h[k:k + 2], 16) for k in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

def returns_chart(returns, holdings, title="Return vs Cost", columns=3):
    order = holdings.sort_values("market_value_eur", ascending=False)["symbol"]
    symbols = [s for s in order if s in returns.columns]
    rows = -(-len(symbols) // columns)
    last = returns.iloc[-1]

    subtitle = ("Each line starts on your purchase date: your shares valued in EUR vs what you paid (exchange rate of the day, fees included)."
                if LEDGER.get("lots") is not None else
                "Today's position valued in EUR vs what you paid. Before your purchase date, the line is hypothetical.")
    fig = make_subplots(
        rows=rows, cols=columns, vertical_spacing=0.18, horizontal_spacing=0.07,
        subplot_titles=[f"{s}   {signed_pct(last[s])}" for s in symbols],
    )
    for i, s in enumerate(symbols):
        r, c = i // columns + 1, i % columns + 1
        ret = returns[s]
        labels = [f"<b>{s}</b><br>{d:%d %b %Y} {d:%H:%M}<br>{signed_pct(v)}" if (d.hour or d.minute) else f"<b>{s}</b><br>{d:%d %b %Y}<br>{signed_pct(v)}"
                  for d, v in zip(ret.index, ret)]
        fig.add_trace(go.Scatter(
            x=ret.index, y=ret.clip(lower=0), mode="none", fill="tozeroy",
            fillcolor=rgba(GREEN_PALETTE[3], 0.35), hoverinfo="skip"), row=r, col=c)
        fig.add_trace(go.Scatter(
            x=ret.index, y=ret.clip(upper=0), mode="none", fill="tozeroy",
            fillcolor=rgba(RED_PALETTE[1], 0.35), hoverinfo="skip"), row=r, col=c)
        fig.add_trace(go.Scatter(
            x=ret.index, y=ret, mode="lines", line=dict(color=holding_color(s), width=0.9),
            customdata=labels, hovertemplate="%{customdata}<extra></extra>"), row=r, col=c)

    fig.add_hline(y=0, line_dash="dot", line_color="#999", row="all", col="all")
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(ticksuffix="%", tickformat=",.0f", gridcolor="#e6e6e6", zeroline=False)
    fig.update_annotations(font_size=13)
    fig.update_layout(
        title=dict(text=f"{title}<br><sup>{subtitle}</sup>"),
        height=300 * rows + 110,
        margin=dict(t=100, b=40, l=60, r=20),
        showlegend=False,
        plot_bgcolor="white",
    )
    return fig


def add_period_buttons(fig, returns, holdings, columns=3, default="1Y"):
    """Period buttons that set the x range AND refit every panel's y range to the visible data."""
    order = holdings.sort_values("market_value_eur", ascending=False)["symbol"]
    symbols = [s for s in order if s in returns.columns]
    last_day = returns.index[-1]
    periods = {
        "1D": last_day - pd.Timedelta(days=1),
        "5D": last_day - pd.Timedelta(days=5),
        "1M": last_day - pd.DateOffset(months=1),
        "3M": last_day - pd.DateOffset(months=3),
        "6M": last_day - pd.DateOffset(months=6),
        "YTD": pd.Timestamp(year=last_day.year, month=1, day=1),
        "1Y": last_day - pd.DateOffset(years=1),
        "All": returns.dropna(how="all").index[0],               # first day any holding existed
    }

    def window(start):
        change = {}
        for i, s in enumerate(symbols):
            suffix = "" if i == 0 else str(i + 1)
            visible = returns.loc[returns.index >= start, s].dropna()
            lo, hi = (min(visible.min(), 0), max(visible.max(), 0)) if len(visible) else (0, 0)   # keep the 0% line in view
            pad = (hi - lo) * 0.08 or 1
            change[f"xaxis{suffix}.range"] = [start, last_day]
            change[f"yaxis{suffix}.range"] = [lo - pad, hi + pad]
        return change

    buttons = [dict(label=name, method="relayout", args=[window(start)]) for name, start in periods.items()]
    fig.update_layout(
        updatemenus=[dict(
            type="buttons", direction="right", buttons=buttons,
            active=list(periods).index(default),
            x=1, y=1.17, xanchor="right", yanchor="bottom", pad=dict(t=0, b=8),
        )],
        margin=dict(t=200, b=40, l=60, r=20), height=fig.layout.height + 50,
    )
    fig.update_xaxes(tickformatstops=[                    # label style follows the zoom level
        dict(dtickrange=[None, 43200000], value="%H:%M<br>%d %b"),   # ticks 12h apart or less -> hours
        dict(dtickrange=[86400000, "M1"], value="%d %b"),            # days
        dict(dtickrange=["M1", None], value="%b '%y"),               # months
    ])
    fig.update_xaxes(matches="x")          # dragging a zoom box in one panel moves all of them
    fig.update_layout(**{k: v for k, v in window(periods[default]).items()})
    return fig

# --- holdings vs IWDA ---
BENCH = "IWDA.AS"
TIERS = [("5y", "1d"), ("730d", "1h"), ("60d", "30m")]   # Yahoo keeps 30m ~60 days, 1h ~2 years
MAX_POINTS = 1500                                          # per line and window, keeps the chart light

def fetch(symbol, period, interval):
    s = cached_close(symbol, period, interval).dropna()
    if interval == "1d":
        s.index = s.index.tz_localize(None).normalize()
    else:
        s.index = s.index.tz_convert("Europe/Paris").tz_localize(None)    # one clock for every exchange
    return s[~s.index.duplicated(keep="last")]

def tier_history(symbol):
    """Daily far back, hourly for ~2 years, 30-minute for the last ~60 days, stitched together."""
    parts, cutoff = [], None
    for period, interval in reversed(TIERS):               # finest first
        s = fetch(symbol, period, interval)
        if s.empty:
            continue
        if cutoff is not None:
            s = s[s.index < (cutoff.normalize() if interval == "1d" else cutoff)]
        if not s.empty:
            parts.append(s)
            cutoff = s.index[0]
    return pd.concat(parts[::-1]).sort_index()

def eur_series(symbol, currency, eurusd):
    p = tier_history(symbol)
    if currency == "EUR":
        return p
    if currency == "USD":
        fx = eurusd.reindex(eurusd.index.union(p.index)).ffill().reindex(p.index)   # rate at that moment
        return (p / fx).dropna()
    raise ValueError(f"No exchange rate set up for {currency} ({symbol})")

def perf(s, start):
    """% change since the first price at or after `start`."""
    w = s[s.index >= start]
    if len(w) < 2:
        return w.iloc[0:0]
    if len(w) > MAX_POINTS:
        w = w.iloc[np.unique(np.linspace(0, len(w) - 1, MAX_POINTS).astype(int))]
    base = s[s.index >= start].iloc[0]
    return (w / base - 1) * 100

def portfolio_index(series):
    """Your portfolio as one line: starts at 100 on the first purchase, shares join on their purchase dates,
    and buying never counts as a gain (only the price moves of what you held just before each moment do)."""
    lots = LEDGER.get("lots")
    if lots is None or lots.empty:
        return None
    symbols = [s for s in series if s in set(lots["symbol"])]
    if not symbols:
        return None
    times = sorted(set().union(*[set(series[s].index) for s in symbols]))
    prices = pd.DataFrame({s: series[s] for s in symbols}).reindex(times).ffill()          # EUR price of each holding
    shares = pd.DataFrame(0.0, index=prices.index, columns=symbols)
    for lot in lots.itertuples():
        if lot.symbol in shares:
            shares.loc[shares.index >= lot.date, lot.symbol] += lot.quantity
    held = shares.shift(1)                                                                    # what you owned just before
    before = (held * prices.shift(1)).sum(axis=1)
    move = (held * prices.diff()).sum(axis=1)
    step = (move / before).where(before > 0, 0.0).fillna(0.0)
    index = (1 + step).cumprod() * 100
    return index[index.index >= shares.sum(axis=1).gt(0).idxmax()]                           # starts on the first purchase

def benchmark_chart(holdings, bench=BENCH, default="1Y", title="Holdings vs IWDA", first_buy=None):
    eurusd = tier_history("EURUSD=X")
    ccy = dict(zip(holdings["symbol"], holdings["currency"]))
    names = [s for s in holdings.sort_values("market_value_eur", ascending=False)["symbol"] if s != bench]
    series = {s: eur_series(s, ccy[s], eurusd) for s in names}
    series[bench] = eur_series(bench, ccy.get(bench, "EUR"), eurusd)
    bname = bench.split(".")[0]
    order = names + [bench]                                   # trace order: holdings, then the benchmark
    colors = {s: holding_color(s) for s in order}
    portfolio = portfolio_index(series)
    if portfolio is not None:                                 # your whole portfolio, in black like the volatility chart
        series["Portfolio"] = portfolio
        order, colors["Portfolio"] = order + ["Portfolio"], PORTFOLIO_COLOR

    first = min(s.index[0] for s in series.values())
    last = max(s.index[-1] for s in series.values())
    periods = {
        "1D": last - pd.Timedelta(days=1), "5D": last - pd.Timedelta(days=5),
        "1M": last - pd.DateOffset(months=1), "3M": last - pd.DateOffset(months=3),
        "6M": last - pd.DateOffset(months=6), "YTD": pd.Timestamp(year=last.year, month=1, day=1),
        "1Y": last - pd.DateOffset(years=1), "All": first,
    }
    if first_buy is not None:                                  # start on the day of your first purchase
        periods = {**{k: v for k, v in periods.items() if k != "All"}, "Since 1st buy": pd.Timestamp(first_buy), "All": first}

    def window(start):
        data = [perf(series[s], start) for s in order]
        x = [p.index.strftime("%Y-%m-%d %H:%M").tolist() for p in data]
        y = [p.round(3).tolist() for p in data]
        legend = [f"{bname if s == bench else s}  {p.iloc[-1]:+.1f}%" if len(p) else f"{s}  n/a"
                  for s, p in zip(order, data)]              # legend shows the return over the period
        vals = pd.concat(data + [pd.Series([0.0])])
        pad = (vals.max() - vals.min()) * 0.08 or 1
        layout = {"xaxis.range": [start, last], "yaxis.range": [vals.min() - pad, vals.max() + pad]}
        return x, y, legend, layout

    fig = go.Figure()
    x0, y0, names0, layout0 = window(periods[default])
    for s, x, y, nm in zip(order, x0, y0, names0):
        label = bname if s == bench else s
        fig.add_trace(go.Scatter(
            x=x, y=y, name=nm, mode="lines",
            line=dict(color=colors[s], width=2.8 if s in (bench, "Portfolio") else 1.8),
            hovertemplate=f"<b>{label}</b><br>%{{x|%d %b %Y %H:%M}}<br>%{{y:+.2f}}%<extra></extra>",
        ))

    buttons = []
    for name, start in periods.items():
        x, y, legend, layout = window(start)
        buttons.append(dict(label=name, method="update", args=[{"x": x, "y": y, "name": legend}, layout]))

    fig.add_hline(y=0, line_dash="dot", line_color="#999")
    fig.update_xaxes(showgrid=False, tickformatstops=[
        dict(dtickrange=[None, 43200000], value="%H:%M<br>%d %b"),   # hours
        dict(dtickrange=[86400000, "M1"], value="%d %b"),            # days
        dict(dtickrange=["M1", None], value="%b '%y"),               # months
    ])
    fig.update_yaxes(ticksuffix="%", tickformat=",.0f", gridcolor="#e6e6e6", zeroline=False)
    fig.update_layout(
        title=dict(text=f"{title}<br><sup>Every line starts at 0% at the beginning of the selected period."
                        + (" Black: your portfolio, starting at your first purchase; new purchases are added without counting as a gain." if portfolio is not None else "") + "</sup>"),
        updatemenus=[dict(type="buttons", direction="right", buttons=buttons,
                          active=list(periods).index(default),
                          x=0, y=1.0, xanchor="left", yanchor="bottom", pad=dict(t=0, b=8))],
        legend=dict(x=1.01, y=1, xanchor="left", yanchor="top", font=dict(size=12), maxheight=1),
        height=520, margin=dict(t=130, b=50, l=60, r=20),
        plot_bgcolor="white",
    )
    fig.update_layout(**layout0)
    return fig

# --- correlation and volatility ---
BENCH = "IWDA.AS"
PORTFOLIO_COLOR = "#000000"   # black
AVERAGE_COLOR = "#7a7a7a"

def daily_closes(symbol, period="5y"):
    s = cached_close(symbol, period, "1d").dropna()
    s.index = s.index.tz_localize(None).normalize()
    return s[~s.index.duplicated(keep="last")]

def daily_returns_eur(holdings, period="5y"):
    """Daily % returns in EUR, only on days when every market was open (same interval for every stock)."""
    symbols = list(holdings["symbol"])
    prices = pd.DataFrame({s: daily_closes(s, period) for s in symbols}).dropna()
    fx = daily_closes("EURUSD=X", period)
    fx = fx.reindex(fx.index.union(prices.index)).ffill().reindex(prices.index)     # USD per EUR
    for s, c in zip(holdings["symbol"], holdings["currency"]):
        if c == "USD":
            prices[s] = prices[s] / fx
        elif c != "EUR":
            raise ValueError(f"No exchange rate set up for {c} ({s})")
    return prices.pct_change().dropna()

def risk_stats(returns, weights):
    """Correlation matrix, annualised volatility (%) per holding, and for the portfolio at today's weights."""
    corr = returns.corr()
    vol = returns.std() * np.sqrt(252) * 100
    w = weights.reindex(returns.columns).values
    portfolio = float(np.sqrt(w @ returns.cov().values @ w) * np.sqrt(252) * 100)
    average = float((w * vol.values).sum())
    return corr, vol, portfolio, average

def correlation_volatility_chart(holdings, default="1Y", title="Correlation and Volatility"):
    order = holdings.sort_values("market_value_eur", ascending=False)
    names = list(order["symbol"])
    weights = order.set_index("symbol")["market_value_eur"]
    weights = weights / weights.sum()
    returns = daily_returns_eur(order)[names]
    last = returns.index[-1]

    others = [s for s in names if s != BENCH]
    bar_colors = [holding_color(s) for s in names]
    periods = {"6M": 6, "1Y": 12, "3Y": 36, "5Y": 60}

    def window(months):
        r = returns[returns.index >= last - pd.DateOffset(months=months)]
        corr, vol, port, avg = risk_stats(r, weights)
        layout = {
            "xaxis2.range": [0, max(vol.max(), port, avg) * 1.15],
            "shapes[0].x0": port, "shapes[0].x1": port,
            "shapes[1].x0": avg, "shapes[1].x1": avg,
            "annotations[2].text": (f"<b style='color:{PORTFOLIO_COLOR}'>Black line</b>: portfolio volatility {port:.1f}%   ·   "
                                    f"<b style='color:{AVERAGE_COLOR}'>Grey line</b>: volatility if all holdings moved in lockstep {avg:.1f}%   ·   "
                                    f"diversification removes <b>{avg - port:.1f} pts</b>"),
        }
        return corr.round(3).values.tolist(), vol.round(2).tolist(), layout

    z0, x0, layout0 = window(periods[default])
    fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42], shared_yaxes=True,
                        horizontal_spacing=0.04,
                        subplot_titles=["Correlation of daily returns (EUR)", "Annual volatility"])
    fig.add_trace(go.Heatmap(
        z=z0, x=names, y=names, zmin=0, zmax=1, showscale=False, xgap=2, ygap=2,
        colorscale=[[0, "#ffffff"], [1, "#1463ab"]],      # white to the capital blue, stopping just short of the exact blue
        texttemplate="%{z:.2f}", textfont=dict(size=13),
        hovertemplate="<b>%{y} / %{x}</b><br>correlation %{z:.2f}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=x0, y=names, orientation="h", marker=dict(color=bar_colors),
        texttemplate="%{x:.1f}%", textposition="outside", cliponaxis=False,
        hovertemplate="<b>%{y}</b><br>annual volatility %{x:.1f}%<extra></extra>",
    ), row=1, col=2)

    for key, col in (("portfolio", PORTFOLIO_COLOR), ("average", AVERAGE_COLOR)):
        fig.add_shape(type="line", xref="x2", yref="paper", y0=0, y1=1, x0=0, x1=0,
                      line=dict(color=col, width=2.5 if key == "portfolio" else 1.8,
                                dash="solid" if key == "portfolio" else "dot"))
    fig.add_annotation(xref="paper", yref="paper", x=0, y=-0.17, xanchor="left", yanchor="top",
                       showarrow=False, font=dict(size=12), text="")
    fig.add_annotation(xref="paper", yref="paper", x=0, y=-0.27, xanchor="left", yanchor="top",
                       showarrow=False, align="left", font=dict(size=11, color="#6b6b6b"),
                       text=("<b>Portfolio volatility</b> is how much the total value of your portfolio typically moves over a year "
                             "(the annualised standard deviation of its daily returns, at today's weights).<br>"))

    buttons = []
    for label, months in periods.items():
        z, x, layout = window(months)
        # one value per trace: the heatmap takes z, the bars take x (the other attribute is simply ignored)
        buttons.append(dict(label=label, method="update",
                            args=[{"z": [z, z], "x": [names, x]}, layout]))

    fig.update_xaxes(showgrid=False, row=1, col=1)
    fig.update_xaxes(showgrid=True, gridcolor="#e6e6e6", ticksuffix="%", zeroline=False, row=1, col=2)
    fig.update_yaxes(autorange="reversed", showgrid=False)
    fig.update_annotations(font_size=13)
    fig.update_layout(
        title=dict(text=title),
        updatemenus=[dict(type="buttons", direction="right", buttons=buttons,
                          active=list(periods).index(default),
                          x=1, y=1.13, xanchor="right", yanchor="bottom", pad=dict(t=0, b=8))],
        height=580, margin=dict(t=140, b=150, l=70, r=40),
        plot_bgcolor="white", showlegend=False,
    )
    fig.update_layout(**layout0)
    return fig

# ---------- step 1: ask for the CSV ----------
REQUIRED_COLUMNS = ["symbol", "shares", "currency", "cost_basis_eur"]      # dividends_eur is optional

def ask_for_csv():
    found = sorted(glob.glob("holdings-*.csv"))                  # newest first by the date in the name
    default = found[-1] if found else None
    while True:
        hint = f" [Enter = {default}]" if default else ""
        answer = input(f"Path to your holdings CSV{hint}: ").strip().strip("'\"")
        if not answer and default:
            answer = default
        path = os.path.expanduser(answer)
        if os.path.isfile(path):
            return path
        print(f"  File not found: {answer!r}. Try again.")

# ---------- cash: read from cash-YYYY-MM-DD.csv, next to holdings-YYYY-MM-DD.csv ----------
def load_cash_eur(holdings_path, rates):
    folder, name = os.path.split(holdings_path)
    cash_path = os.path.join(folder, name.replace("holdings-", "cash-", 1)) if name.startswith("holdings-") else None
    if not cash_path or not os.path.isfile(cash_path):
        return 0.0                                           # no cash file: no cash slice
    cash = pd.read_csv(cash_path)
    unknown = set(cash["currency"]) - set(rates)
    if unknown:
        raise ValueError(f"No exchange rate set up for cash in: {', '.join(sorted(unknown))}")
    return float((cash["amount"] * cash["currency"].map(rates)).sum())

# ---------- step 2: from the CSV to every number the charts need ----------
def load_portfolio(path):
    _download_close.cache_clear()                                # always start from fresh prices
    _dividends.cache_clear()
    holdings = pd.read_csv(path)
    missing = [c for c in REQUIRED_COLUMNS if c not in holdings.columns]
    if missing:
        raise ValueError(f"The CSV is missing the column(s): {', '.join(missing)}")
    if "dividends_eur" not in holdings.columns:                  # the bank's Excel has no dividends
        holdings["dividends_eur"] = 0.0
    holdings = apply_ledger(holdings, path)                      # real cost (own-day rate, fees) + dividends, if a ledger exists

    prices = {s: cached_close(s, "5d").dropna().iloc[-1] for s in holdings["symbol"]}
    eurusd = cached_close("EURUSD=X", "5d").dropna().iloc[-1]
    rates = {"EUR": 1.0, "USD": 1 / eurusd}
    cash_eur = load_cash_eur(path, rates)
    unknown = set(holdings["currency"]) - set(rates)
    if unknown:
        raise ValueError(f"No exchange rate set up for: {', '.join(sorted(unknown))}")

    holdings["price"] = holdings["symbol"].map(prices)
    holdings["fx_to_eur"] = holdings["currency"].map(rates)
    holdings["market_value_eur"] = holdings["shares"] * holdings["price"] * holdings["fx_to_eur"]
    holdings["gain_eur"] = holdings["market_value_eur"] - holdings["cost_basis_eur"]
    holdings["gain_pct"] = holdings["gain_eur"] / holdings["cost_basis_eur"] * 100
    holdings["weight_pct"] = holdings["market_value_eur"] / holdings["market_value_eur"].sum() * 100
    add_return_columns(holdings)

    quote_types = {s: yf.Ticker(s).info.get("quoteType") for s in holdings["symbol"]}
    holdings["asset_type"] = holdings["symbol"].map(quote_types).map({"EQUITY": "Equities", "ETF": "ETFs"}).fillna("Other")
    allocation = holdings.groupby("asset_type")["market_value_eur"].sum().sort_values(ascending=False)

    # sector exposure: stocks in their own sector, ETFs split with their published sector weights
    def clean(label):
        return label.lower().replace(" ", "").replace("_", "")
    sector_names = {
        "technology": "Technology", "communicationservices": "Communication Services",
        "consumercyclical": "Consumer Cyclical", "consumerdefensive": "Consumer Defensive",
        "financialservices": "Financial Services", "healthcare": "Healthcare",
        "industrials": "Industrials", "basicmaterials": "Basic Materials",
        "energy": "Energy", "utilities": "Utilities", "realestate": "Real Estate",
        "unknown": "Unknown",
    }
    equities = holdings[holdings["asset_type"] == "Equities"].copy()
    equities["sector"] = [clean(yf.Ticker(s).info.get("sector") or "Unknown") for s in equities["symbol"]]
    rows = [equities[["sector", "market_value_eur"]]]
    etf_rows = []
    for _, row in holdings[holdings["asset_type"] == "ETFs"].iterrows():
        for sector, weight in yf.Ticker(row["symbol"]).funds_data.sector_weightings.items():
            etf_rows.append({"sector": clean(sector), "market_value_eur": row["market_value_eur"] * weight})
    rows.append(pd.DataFrame(etf_rows))
    sector_totals = pd.concat(rows).groupby("sector")["market_value_eur"].sum().sort_values(ascending=False)
    sector_totals.index = sector_totals.index.map(lambda k: sector_names.get(k, k.title()))
    return holdings, allocation, sector_totals, cash_eur

# ---------- step 3: show everything ----------
def section(text):
    display(Markdown(f"## {text}"))

def show_dashboard(path):
    holdings, allocation, sector_totals, cash = load_portfolio(path)
    assign_holding_colors(holdings)
    invested = holdings["market_value_eur"].sum()
    total = invested + cash

    money = f"€{invested:,.0f} invested + €{cash:,.0f} cash = €{total:,.0f}" if cash > 0 else f"€{total:,.0f}"
    display(Markdown(f"# Portfolio dashboard\n**{len(holdings)} holdings · {money} · source: `{os.path.basename(path)}`**"))
    columns = ["symbol", "shares", "currency", "price", "market_value_eur", "cost_basis_eur", "gain_eur", "gain_pct", "weight_pct"]
    if LEDGER.get("lots") is not None:
        columns = ["symbol", "first_buy", "shares", "currency", "price", "market_value_eur", "cost_basis_eur", "fees_eur",
                   "gain_eur", "gain_pct", "dividends_eur", "total_return_pct", "xirr_pct", "weight_pct"]
    table = holdings[columns].copy()
    numbers = table.select_dtypes("number").columns
    table[numbers] = table[numbers].round(2)
    if "first_buy" in table:
        table["first_buy"] = table["first_buy"].dt.strftime("%Y-%m-%d")
    display(table)
    for note in LEDGER.get("notes", []):
        display(Markdown(f"*{note}*"))
    if "portfolio_xirr" in LEDGER:
        display(Markdown(f"**Since your first purchase ({LEDGER['first_buy']:%d %b %Y}): {signed_pct(LEDGER['portfolio_total_return'])} "
                         f"including net dividends, {signed_pct(LEDGER['portfolio_xirr'])} a year.**"))

    section("Allocation")
    with_cash = pd.concat([allocation, pd.Series({"Cash": cash})]) if cash > 0 else allocation
    allocation_bar(with_cash, "Asset Allocation", colors=ASSET_COLORS).show()
    allocation_bar(sector_totals, "Sector Allocation", colors=SECTOR_COLORS).show()
    slices = holdings.set_index("symbol")["market_value_eur"]
    if cash > 0:
        slices = pd.concat([slices, pd.Series({"Cash": cash})])
    slice_colors = {**{sym: holding_color(sym) for sym in holdings["symbol"]}, "Cash": ASSET_COLORS["Cash"]}
    doughnut(slices, "Portfolio Holdings", fixed_colors=slice_colors).show()
    currency_bar(currency_exposure(holdings)).show()

    section("Gains and losses")
    gain_loss_chart(holdings).show()
    cost_to_value_chart(holdings).show()

    section("Performance over time")
    prices, eurusd = load_history(holdings["symbol"])
    returns = holding_returns(holdings, prices, eurusd)
    add_period_buttons(returns_chart(returns, holdings), returns, holdings).show()
    benchmark_chart(holdings, first_buy=LEDGER.get("first_buy")).show()

    section("Risk")
    correlation_volatility_chart(holdings).show()


# ---------- paste to CSV: the platform's text becomes holdings-YYYY-MM-DD.csv ----------
def _is_number(token):
    return bool(re.fullmatch(r"[+-]?[\d,]*\.?\d+%?|--", token))

def _num(token):
    if token == "--":                                   # the platform's way of writing "nothing"
        return 0.0
    return float(token.replace(",", "").replace("+", "").replace("%", ""))

def parse_paste(text):
    """Read the platform's text: a symbol line, then one row of numbers, for each holding; then 'Total Cash'."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    rows, cash, i = [], None, 0
    while i < len(lines):
        line = lines[i]
        if line.lower().startswith("total cash"):
            rest = line[len("total cash"):].split()
            if rest:
                cash, i = _num(rest[0]), i + 1
            elif i + 1 < len(lines):
                cash, i = _num(lines[i + 1].split()[0]), i + 2
            else:
                raise ValueError("'Total Cash' is in the paste but its amount is missing.")
        elif re.fullmatch(r"[A-Z0-9][A-Z0-9.\-^=]*", line) and re.search(r"[A-Z]", line):
            if i + 1 >= len(lines):
                raise ValueError(f"{line}: the row of numbers after this symbol is missing.")
            tokens = lines[i + 1].split()
            while tokens and not _is_number(tokens[0]):  # drop the status word ('Open')
                tokens.pop(0)
            if len(tokens) < 10:
                raise ValueError(f"{line}: expected at least 10 numbers after the status word, found {len(tokens)}.")
            v = [_num(t) for t in tokens[:10]]
            rows.append(dict(symbol=line, shares=v[0], last_price=v[1], avg_cost=v[2], cost_basis_eur=v[3],
                             market_value_eur=v[4], dividends_eur=v[5], day_pct=v[6], day_change=v[7],
                             total_pct=v[8], total_gain_eur=v[9]))
            i += 2
        else:
            raise ValueError(f"Could not read this line of the paste: {line!r}\n"
                             "Expected a symbol on its own line, then a row such as: Open  23  130.19  108.00  2,486.00  ...")
    if not rows:
        raise ValueError("No holdings found. Paste your portfolio text between the triple quotes.")
    return pd.DataFrame(rows), cash

def check_paste(df):
    """Checks that catch a shifted column or a half-copied paste. Returns a list of problems."""
    problems = []
    if df["symbol"].duplicated().any():
        problems.append("Duplicate symbols: " + ", ".join(df.loc[df["symbol"].duplicated(), "symbol"]))
    for r in df.itertuples():
        gap = r.market_value_eur - r.cost_basis_eur
        if abs(gap - r.total_gain_eur) > 0.02:
            problems.append(f"{r.symbol}: market value - cost basis = {gap:,.2f}, but the paste shows a total gain of {r.total_gain_eur:,.2f}")
        pct = r.total_gain_eur / r.cost_basis_eur * 100
        if abs(pct - r.total_pct) > 0.02:
            problems.append(f"{r.symbol}: total gain / cost basis = {pct:.2f}%, but the paste shows {r.total_pct:.2f}%")
        if r.shares <= 0:
            problems.append(f"{r.symbol}: shares should be above zero (found {r.shares:g})")
    return problems

def lookup_currency(symbol):
    ticker = yf.Ticker(symbol)
    try:
        currency = ticker.fast_info["currency"]
    except Exception:
        currency = None
    if not currency:
        currency = ticker.info.get("currency")
    if not currency:
        raise ValueError(f"Yahoo did not return a currency for {symbol}. Check the symbol.")
    return currency.upper()

def _write_if_ok(path, text):
    """Write the file; if it already exists with different content, ask before overwriting."""
    if os.path.exists(path):
        if open(path, encoding="utf-8").read() == text:
            print(f"  {path}: already up to date.")
            return
        answer = input(f"  {path} already exists with different content. Overwrite? [y/N]: ").strip().lower()
        if answer != "y":
            raise FileExistsError(f"{path} was not overwritten. Rename or move it, or run again and answer y.")
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(text)
    print(f"  Saved {path}")

def paste_to_csv(paste, as_of=None, folder="."):
    """Platform text -> holdings-YYYY-MM-DD.csv (+ cash-YYYY-MM-DD.csv). Returns the holdings CSV path."""
    df, cash = parse_paste(paste)
    problems = check_paste(df)
    if problems:
        raise ValueError("The paste did not pass its checks, nothing was saved:\n  - " + "\n  - ".join(problems))
    df["currency"] = [lookup_currency(s) for s in df["symbol"]]

    as_of = as_of or dt.date.today().isoformat()
    lines = ["symbol,shares,avg_cost,currency,cost_basis_eur,dividends_eur,as_of"]
    for r in df.itertuples():
        shares = int(r.shares) if r.shares == int(r.shares) else r.shares
        dividends = "0" if r.dividends_eur == 0 else f"{r.dividends_eur:.2f}"
        lines.append(f"{r.symbol},{shares},{r.avg_cost:.2f},{r.currency},{r.cost_basis_eur:.2f},{dividends},{as_of}")
    holdings_path = os.path.join(folder, f"holdings-{as_of}.csv")

    print(f"Read {len(df)} holdings; every check passed (market value - cost basis = total gain, gain %).")
    display(df[["symbol", "shares", "currency", "avg_cost", "cost_basis_eur", "market_value_eur", "dividends_eur"]])
    _write_if_ok(holdings_path, "\n".join(lines) + "\n")
    if cash is None:
        print("  No 'Total Cash' line in the paste, so no cash file was written.")
    else:
        print("  Cash is assumed to be in EUR (the paste does not say).")
        _write_if_ok(os.path.join(folder, f"cash-{as_of}.csv"),
                     f"currency,amount,as_of\nEUR,{cash:.2f},{as_of}\n")
    return holdings_path


# ---------- the bank's Excel export -> holdings-YYYY-MM-DD.csv + cash-YYYY-MM-DD.csv ----------
# Sheets and columns are found by POSITION (not by name), so the export keeps working if the app changes language.
#   sheet 1 (positions): B market, D symbol, E quantity, G average cost (own currency), J initial value (own currency),
#                        H last price, O initial value in EUR
#   sheet 3 (cash):      C currency, D amount
MARKET_SUFFIX = {"paris": ".PA", "amsterdam": ".AS", "us": ""}   # market name in column B -> Yahoo suffix (add yours here)

def _real_excels(folder):
    """.xlsx files in a folder, ignoring the temporary '~$' lock files Excel leaves while a workbook is open."""
    return [p for p in glob.glob(os.path.join(os.path.expanduser(folder), "*.xlsx")) if not os.path.basename(p).startswith("~$")]

def _looks_like_bank_excel(path):
    """The bank's export, recognised by its layout (3+ sheets, 15+ columns on sheet 1, 4+ on sheet 3), never by its name."""
    try:
        sheets = list(pd.read_excel(path, sheet_name=None).values())
    except Exception:
        return False
    return len(sheets) >= 3 and sheets[0].shape[1] >= 15 and len(sheets[0]) > 0 and sheets[2].shape[1] >= 4

def find_bank_excel(required=True):
    """Newest Excel in the project folder or Downloads that has the bank export's layout. No file name, no prompt."""
    candidates = sorted(_real_excels(".") + _real_excels(DOWNLOADS), key=os.path.getmtime, reverse=True)
    for path in candidates:
        if _looks_like_bank_excel(path):
            print(f"  Using {path}")
            return path
    if not required:
        return None
    raise FileNotFoundError("No portfolio export found. Looked at: " + (", ".join(os.path.basename(p) for p in candidates) or "no .xlsx file")
                            + " (project folder and Downloads). Export it again from the banking app.")

def read_bank_excel(xlsx_path):
    """Read sheet 1 and sheet 3 by position. Returns (holdings DataFrame, cash DataFrame)."""
    sheets = list(pd.read_excel(os.path.expanduser(xlsx_path), sheet_name=None).values())
    if len(sheets) < 3:
        raise ValueError(f"Expected at least 3 sheets (positions, orders, cash), found {len(sheets)}.")
    pos, cash = sheets[0], sheets[2]
    if pos.shape[1] < 15:
        raise ValueError(f"Sheet 1 should have at least 15 columns, found {pos.shape[1]}. Did the export layout change?")
    if cash.shape[1] < 4:
        raise ValueError(f"Sheet 3 should have at least 4 columns, found {cash.shape[1]}. Did the export layout change?")
    pos = pos[pos.iloc[:, 3].notna()]
    holdings = pd.DataFrame({
        "market": pos.iloc[:, 1].astype(str).str.strip(), "ticker": pos.iloc[:, 3].astype(str).str.strip().str.upper(),
        "shares": pd.to_numeric(pos.iloc[:, 4]), "avg_cost": pd.to_numeric(pos.iloc[:, 6]),
        "file_price": pd.to_numeric(pos.iloc[:, 7]), "initial_own": pd.to_numeric(pos.iloc[:, 9]),
        "cost_basis_eur": pd.to_numeric(pos.iloc[:, 14]),
    }).reset_index(drop=True)
    cash = cash[cash.iloc[:, 2].notna()]
    cash = pd.DataFrame({"currency": cash.iloc[:, 2].astype(str).str.strip().str.upper(),
                         "amount": pd.to_numeric(cash.iloc[:, 3])}).reset_index(drop=True)
    return holdings, cash

def excel_to_csv(xlsx_path, folder="."):
    """Bank Excel -> holdings-YYYY-MM-DD.csv (+ cash-YYYY-MM-DD.csv). Returns the holdings CSV path."""
    h, cash = read_bank_excel(xlsx_path)
    if h.empty:
        raise ValueError("No positions found on sheet 1.")
    unknown = sorted(set(h["market"].str.lower()) - set(MARKET_SUFFIX))
    if unknown:
        raise ValueError("Unknown market(s) in column B: " + ", ".join(unknown) +
                         ". Add them to MARKET_SUFFIX in the setup cell (e.g. 'paris': '.PA').")
    h["symbol"] = h["ticker"] + h["market"].str.lower().map(MARKET_SUFFIX)

    problems = []
    if h["symbol"].duplicated().any():
        problems.append("Duplicate symbols: " + ", ".join(h.loc[h["symbol"].duplicated(), "symbol"]))
    for r in h.itertuples():
        if r.shares <= 0 or r.cost_basis_eur <= 0:
            problems.append(f"{r.symbol}: quantity and cost must be above zero")
        if abs(r.shares * r.avg_cost - r.initial_own) > 0.05:
            problems.append(f"{r.symbol}: quantity x average cost = {r.shares * r.avg_cost:,.2f}, but the file shows an initial value of {r.initial_own:,.2f}")
    if problems:
        raise ValueError("The Excel did not pass its checks, nothing was saved:\n  - " + "\n  - ".join(problems))

    h["currency"] = [lookup_currency(s) for s in h["symbol"]]
    for r in h.itertuples():          # a wrong suffix would give another stock's price: compare with the file's price
        last = yf.Ticker(r.symbol).history(period="5d")["Close"].dropna().iloc[-1]
        if pd.notna(r.file_price) and r.file_price > 0 and abs(last / r.file_price - 1) > 0.10:
            problems.append(f"{r.symbol}: Yahoo's price is {last:,.2f} but the file says {r.file_price:,.2f}. Wrong symbol or market?")
    if problems:
        raise ValueError("The symbols did not match the file's prices, nothing was saved:\n  - " + "\n  - ".join(problems))

    as_of = dt.date.today().isoformat()
    lines = ["symbol,shares,avg_cost,currency,cost_basis_eur,as_of"]
    for r in h.itertuples():
        shares = int(r.shares) if r.shares == int(r.shares) else r.shares
        lines.append(f"{r.symbol},{shares},{r.avg_cost:.2f},{r.currency},{r.cost_basis_eur:.2f},{as_of}")
    holdings_path = os.path.join(folder, f"holdings-{as_of}.csv")

    print(f"Read {len(h)} positions and {len(cash)} cash line(s) from the Excel; all checks passed.")
    display(h[["symbol", "shares", "currency", "avg_cost", "cost_basis_eur"]])
    display(cash)
    _write_if_ok(holdings_path, "\n".join(lines) + "\n")
    if cash.empty:
        print("  No cash lines on sheet 3, so no cash file was written.")
    else:
        _write_if_ok(os.path.join(folder, f"cash-{as_of}.csv"),
                     "currency,amount,as_of\n" + "".join(f"{r.currency},{r.amount:.2f},{as_of}\n" for r in cash.itertuples()))
    return holdings_path



# ---------- transaction ledger: transactions-YYYY-MM-DD.csv (buys, fees, dates) built from the bank's History PDFs ----------
# One row per buy or sell. `amount` is everything paid (fees included) for a buy, or received for a sell. `symbol` is the Yahoo symbol, found from the stock's name.
# Dividends are not stored: they are worked out from what you held.
LEDGER_PATTERN = "transactions-*.csv"
BELGIAN_TAX = 0.30                       # withheld by your Belgian broker on every dividend, on top of the tax at source
DIVIDEND_TAX_FILE = "dividend-tax.csv"   # one row per market: the tax withheld in the stock's own country
DIVIDEND_TAX_DEFAULT = [                 # market (Yahoo suffix, "US" = no suffix), country, tax at source, status, note
    ("US", "United States", 0.30, "to check", "30% without a W-8BEN form; 15% with one"),
    (".PA", "France", 0.25, "to check", ""),
    (".AS", "Netherlands", 0.15, "to check", ""),
    (".BR", "Belgium", 0.0, "to check", "the Belgian 30% is the only tax"),
    (".DE", "Germany", 0.26375, "to check", "statutory rate"),
    (".MI", "Italy", 0.26, "to check", "statutory rate"),
    (".MC", "Spain", 0.19, "to check", "statutory rate"),
    (".L", "United Kingdom", 0.0, "to check", "no withholding on UK dividends"),
    (".SW", "Switzerland", 0.35, "to check", "statutory rate"),
    (".CO", "Denmark", 0.27, "to check", "statutory rate"),
    (".ST", "Sweden", 0.30, "to check", "statutory rate"),
    (".OL", "Norway", 0.25, "to check", "statutory rate"),
    (".HE", "Finland", 0.35, "to check", "statutory rate for non-residents"),
    (".LS", "Portugal", 0.25, "to check", "statutory rate"),
    (".VI", "Austria", 0.275, "to check", "statutory rate"),
    (".IR", "Ireland", 0.25, "to check", "statutory rate"),
]
TAX_COLUMNS = ["market", "country", "source_tax", "status", "note"]   # status: from your payouts / learned / to check / manual (never changed by the code)
LEDGER = {}                                                       # filled by load_portfolio: lots, dividend events, notes

_NUM = r"-?\d+(?:[ \u00a0\u202f]\d{3})*(?:,\d+)?"
_ROW = re.compile(
    r"(\d{2}/\d{2}/\d{4})\s+(\d{2}/\d{2}/\d{4})\s+(Buy|Sell|Dividend / Interest)\s+(.+?)\s+(\d+(?:,\d+)?)\s+"
    rf"({_NUM})\s([A-Z]{{3}})\s+(\d+,\d+)\s+({_NUM})\s([A-Z]{{3}})\s+({_NUM})\s[A-Z]{{3}}")

def _fr(text):
    return float(re.sub(r"[ \u00a0\u202f]", "", text).replace(",", "."))

def parse_history_pdf(path):
    """Read one 'History' PDF export (French number format) into ledger rows."""
    from pypdf import PdfReader
    text = " ".join((page.extract_text() or "") for page in PdfReader(os.path.expanduser(path)).pages)
    text = re.sub(r"\s+", " ", text)
    rows = []
    for m in _ROW.finditer(text):
        trade_date, _, kind, name, qty, price, ccy, fx, amount, _, _ = m.groups()
        rows.append(dict(
            date=pd.to_datetime(trade_date, format="%d/%m/%Y").strftime("%Y-%m-%d"),
            type={"Buy": "buy", "Sell": "sell", "Dividend / Interest": "dividend"}[kind],
            ticker=name.split()[-1], name=" ".join(name.split()[:-1]), quantity=_fr(qty), price=round(_fr(price), 4), currency=ccy,
            fx=round(_fr(fx), 6), amount=round(abs(_fr(amount)), 2)))
    if not rows:
        raise ValueError(f"No transactions found in {os.path.basename(path)}. Is it the bank's History export?")
    return pd.DataFrame(rows)

def read_ledger(path):
    ledger = pd.read_csv(path)
    ledger["date"] = pd.to_datetime(ledger["date"])
    return ledger

def latest_ledger(folder="."):
    found = sorted(glob.glob(os.path.join(folder, LEDGER_PATTERN)))
    return found[-1] if found else None

def _history_pdfs():
    """PDFs in the DOWNLOADS folder whose name starts with 'History' (any case, any suffix such as ' (1)' or ' Oct')."""
    folder = os.path.expanduser(DOWNLOADS)
    return sorted(p for p in glob.glob(os.path.join(folder, "*.pdf")) if os.path.basename(p).lower().startswith("history"))

LEDGER_KEYS = ["date", "type", "ticker", "quantity", "price", "currency", "fx", "amount"]
SUFFIX_PREFERENCE = [".PA", ".AS", ".BR", ".DE", ".MI", ".MC", ".L", ""]       # tie-break between listings in the same currency

def _known_symbols(folder="."):
    """ticker -> Yahoo symbol, from every holdings file saved so far."""
    known = {}
    for path in sorted(glob.glob(os.path.join(folder, "holdings-*.csv"))):
        try:
            for symbol in pd.read_csv(path)["symbol"]:
                known[symbol.split(".")[0].upper()] = symbol
        except Exception:
            pass
    return known

def find_symbol(name, ticker, currency):
    """Yahoo symbol for a stock the PDF names. Uses the full name to search, the ticker and the currency to pick."""
    try:
        quotes = yf.Search(name, max_results=10).quotes
    except Exception:
        quotes = []
    symbols = [q["symbol"] for q in quotes if q.get("symbol") and q.get("quoteType") in ("EQUITY", "ETF")]
    same_ticker = [x for x in symbols if x.split(".")[0].upper() == ticker.upper()]
    in_currency = []
    for symbol in same_ticker:                                   # the right listing is the one quoted in the PDF's currency
        try:
            if lookup_currency(symbol) == currency:
                in_currency.append(symbol)
        except Exception:
            pass
    if in_currency:
        def rank(x):
            suffix = "." + x.split(".")[1] if "." in x else ""
            return SUFFIX_PREFERENCE.index(suffix) if suffix in SUFFIX_PREFERENCE else len(SUFFIX_PREFERENCE)
        choice = sorted(in_currency, key=rank)[0]
        print(f"  {name} ({ticker}) -> {choice}" + (f"   [other listings: {', '.join(x for x in in_currency if x != choice)}]" if len(in_currency) > 1 else ""))
        return choice
    print(f"  Could not find the Yahoo symbol for {name} ({ticker}, {currency}) automatically.")
    if symbols:
        print("  Yahoo suggests: " + ", ".join(symbols[:6]))
    while True:
        answer = input("  Type its Yahoo symbol (for example AI.PA): ").strip().upper()
        if answer:
            return answer

def fill_symbols(ledger, folder="."):
    """Give every ledger row its Yahoo symbol: from rows already resolved, from earlier holdings files, else from the name."""
    if "symbol" not in ledger:
        ledger["symbol"] = pd.Series([None] * len(ledger), index=ledger.index, dtype="object")
    if "name" not in ledger:
        ledger["name"] = pd.Series([None] * len(ledger), index=ledger.index, dtype="object")
    known = _known_symbols(folder)
    for ticker in ledger.loc[ledger["symbol"].isna(), "ticker"].unique():
        rows = ledger[ledger["ticker"] == ticker]
        done = rows["symbol"].dropna()
        if len(done):
            symbol = done.iloc[0]
        elif ticker.upper() in known:
            symbol = known[ticker.upper()]
        else:
            names = rows["name"].dropna()
            symbol = find_symbol(names.iloc[0] if len(names) else ticker, ticker, rows["currency"].iloc[0])
        ledger.loc[ledger["ticker"] == ticker, "symbol"] = symbol
    names = ledger.dropna(subset=["name"]).groupby("ticker")["name"].first()
    ledger["name"] = ledger["name"].fillna(ledger["ticker"].map(names))
    return ledger

def _learn_from(dividend_lines, ledger, folder):
    """Check the dividend lines of the PDFs against the tax table (only stocks whose Yahoo symbol is known)."""
    if dividend_lines is None or dividend_lines.empty:
        return
    symbol_of = {**{t: s for t, s in _known_symbols(folder).items()}, **ledger.dropna(subset=["symbol"]).groupby("ticker")["symbol"].first().to_dict()}
    learn_dividend_tax(dividend_lines, symbol_of, folder)

def import_history_pdfs(pattern=None, folder="."):
    """Add the buys and sells found in the History PDFs to the ledger. Old rows are kept, duplicates ignored.
    Writes transactions-YYYY-MM-DD.csv only when there is something new."""
    pdfs = sorted(glob.glob(os.path.expanduser(pattern))) if pattern else _history_pdfs()
    old_path = latest_ledger(folder)
    old = read_ledger(old_path).assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d")) if old_path else None
    frames = []
    for p in pdfs:
        try:
            frames.append(parse_history_pdf(p))
        except Exception as error:
            print(f"  Skipped {os.path.basename(p)}: {error}")
    new = pd.concat(frames, ignore_index=True) if frames else None
    dividend_lines = new[new["type"] == "dividend"] if new is not None else None      # not kept, but they show the real tax withheld
    if new is not None:
        new = new[new["type"] != "dividend"]                   # dividends are not kept: they are worked out from what you held
    base = old if old is not None else (new.iloc[0:0] if new is not None else None)
    if base is None:
        raise FileNotFoundError("No transactions yet and no History PDF found in Downloads (the file names must start with 'History'). See 'First time? Start here' at the top of the notebook.")
    merged = pd.concat([base] + ([new] if new is not None else []), ignore_index=True)
    merged = merged[merged["type"] != "dividend"].drop_duplicates(subset=LEDGER_KEYS, keep="first")
    added = merged.merge(base[LEDGER_KEYS].drop_duplicates(), on=LEDGER_KEYS, how="left", indicator=True).query("_merge == 'left_only'")
    needs_symbols = "symbol" not in merged or merged["symbol"].isna().any()
    if added.empty and not needs_symbols:
        print(f"  Transactions are up to date ({len(merged)} buys/sells, read from {len(pdfs)} PDF(s)).")
        _learn_from(dividend_lines, merged, folder)
        return old_path
    merged = fill_symbols(merged.reset_index(drop=True), folder)
    _learn_from(dividend_lines, merged, folder)
    merged = merged.sort_values(["date", "ticker", "type"], ascending=[False, True, True]).reset_index(drop=True)
    if len(added):
        print(f"  {len(added)} new buy/sell row(s) found in the History PDFs; the ledger will have {len(merged)}:")
        display(merged.merge(added[LEDGER_KEYS], on=LEDGER_KEYS)[["date", "type", "name", "symbol", "quantity", "price", "currency", "amount"]])
    else:
        print("  Adding the Yahoo symbol to each transaction (done once).")
    columns = ["date", "type", "ticker", "name", "symbol", "quantity", "price", "currency", "fx", "amount"]
    out = os.path.join(folder, f"transactions-{dt.date.today().isoformat()}.csv")
    _write_if_ok(out, merged[columns].to_csv(index=False))
    return out

def _lots_from_ledger(ledger, symbols):
    """Buys of the symbols you hold, with cost in EUR at the exchange rate of the day, fees included."""
    ledger = ledger.copy()
    ticker_of = {s.split(".")[0]: s for s in symbols}
    if "symbol" not in ledger:
        ledger["symbol"] = pd.Series([None] * len(ledger), index=ledger.index, dtype="object")
    ledger["symbol"] = ledger["symbol"].fillna(ledger["ticker"].map(ticker_of))
    mine = ledger[ledger["symbol"].isin(symbols)]
    sold = mine[mine["type"] == "sell"]
    if len(sold):
        raise ValueError("The ledger contains sales of " + ", ".join(sorted(set(sold["ticker"]))) +
                         ", which the cost calculation does not handle yet.")
    buys = mine[mine["type"] == "buy"].copy()
    buys["cost_eur"] = buys["amount"] / buys["fx"]
    buys["fees_eur"] = (buys["amount"] - buys["quantity"] * buys["price"]) / buys["fx"]
    return buys[["symbol", "date", "quantity", "price", "currency", "cost_eur", "fees_eur"]].sort_values("date").reset_index(drop=True)

@lru_cache(maxsize=None)
def _dividends(symbol):
    d = yf.Ticker(symbol).dividends
    if d is None or len(d) == 0:
        return pd.Series(dtype=float)
    if d.index.tz is not None:
        d = d.index.tz_localize(None).to_series().pipe(lambda i: pd.Series(d.values, index=i.dt.normalize().values))
    return d.astype(float)

def _market_key(symbol):
    """'.PA' for AI.PA, 'US' for a symbol without a suffix."""
    return "." + symbol.split(".")[-1] if "." in symbol else "US"

def load_dividend_tax(folder="."):
    """The tax-at-source table, indexed by market. Created with common countries the first time."""
    path = os.path.join(folder, DIVIDEND_TAX_FILE)
    if not os.path.isfile(path):
        pd.DataFrame(DIVIDEND_TAX_DEFAULT, columns=TAX_COLUMNS).to_csv(path, index=False)
        print(f"  Created {DIVIDEND_TAX_FILE}: tax withheld at source per market. Edit it any time; set status to 'manual' to protect a row.")
    table = pd.read_csv(path, keep_default_na=False)
    table["source_tax"] = pd.to_numeric(table["source_tax"])
    return table.set_index("market")

def _kept_share(symbol, table):
    """(share of a gross dividend that reaches your account, whether the market was found in the table)."""
    key = _market_key(symbol)
    if key in table.index:
        return (1 - table.loc[key, "source_tax"]) * (1 - BELGIAN_TAX), True
    return (1 - 0.30) * (1 - BELGIAN_TAX), False               # unknown market: assume 30% at source

def learn_dividend_tax(dividends, symbol_of, folder="."):
    """Real tax at source per market, from the dividend lines in the PDFs: net received / gross, with the Belgian tax removed."""
    d = dividends[dividends["quantity"] * dividends["price"] > 0].copy()
    d["symbol"] = d["ticker"].map(symbol_of)
    d = d.dropna(subset=["symbol"])
    if d.empty:
        return
    d["market"] = d["symbol"].map(_market_key)
    d["source_tax"] = 1 - (d["amount"] / (d["quantity"] * d["price"])) / (1 - BELGIAN_TAX)
    learned = d.groupby("market")["source_tax"].median().clip(lower=0).round(3)
    table = load_dividend_tax(folder)
    changes = []
    for market, tax in learned.items():
        if market not in table.index:
            tickers = ", ".join(sorted(d.loc[d["market"] == market, "ticker"].unique()))
            changes.append((market, f"market {market} ({tickers})", tax, "learned", "from your dividend payouts", "new"))
            continue
        row = table.loc[market]
        if row["status"] == "manual":
            continue
        same = abs(row["source_tax"] - tax) < 0.01        # cent rounding on small dividends moves the rate by a few tenths of a point
        if same and row["status"] != "to check":
            continue
        changes.append((market, row["country"], row["source_tax"] if same else tax, "learned" if row["status"] in ("to check", "learned") else row["status"],
                        row["note"], "confirmed" if same else f"{row['source_tax']:.1%} ({row['status']})"))
    if not changes:
        return
    print(f"  The dividend lines in your PDFs, compared with {DIVIDEND_TAX_FILE}:")
    for market, country, tax, status, note, what in changes:
        print(f"    {market:4s} {country}: " + ("new row" if what == "new" else "confirmed by your payouts" if what == "confirmed" else f"was {what}, your payouts show") + f" {tax:.1%}")
    if input(f"  Update {DIVIDEND_TAX_FILE}? [y/N]: ").strip().lower() != "y":
        return
    for market, country, tax, status, note, _ in changes:
        table.loc[market] = [country, tax, status, note]
    table.reset_index().to_csv(os.path.join(folder, DIVIDEND_TAX_FILE), index=False)
    print(f"  Saved {DIVIDEND_TAX_FILE}.")

def apply_ledger(holdings, path):
    """Replace the Excel's cost with the real one (own-day exchange rate, fees) and fill in dividends from Yahoo."""
    LEDGER.clear()
    ledger_path = latest_ledger(os.path.dirname(os.path.abspath(path)))
    if ledger_path is None:
        LEDGER["notes"] = ["No `transactions-*.csv` next to the holdings file: cost taken from the Excel (today's exchange rate, no fees), no dividends."]
        return holdings
    ledger = read_ledger(ledger_path)
    lots = _lots_from_ledger(ledger, list(holdings["symbol"]))
    held = lots.groupby("symbol")["quantity"].sum()
    bad = [s for s, q in zip(holdings["symbol"], holdings["shares"]) if abs(held.get(s, 0) - q) > 1e-6]
    notes = [f"Cost, fees and dates from `{os.path.basename(ledger_path)}`."]
    if bad:
        notes.append("**Not matched with the ledger** (quantity differs from the Excel), so they keep the Excel's cost and have no dividends: "
                     + ", ".join(bad) + ". Export the History PDF for them and import it.")
        lots = lots[~lots["symbol"].isin(bad)]
    good = [s for s in holdings["symbol"] if s not in bad]

    holdings["cost_basis_excel_eur"] = holdings["cost_basis_eur"]
    cost = lots.groupby("symbol")["cost_eur"].sum()
    holdings["cost_basis_eur"] = holdings["symbol"].map(cost).fillna(holdings["cost_basis_eur"])
    holdings["fees_eur"] = holdings["symbol"].map(lots.groupby("symbol")["fees_eur"].sum()).fillna(0.0)
    holdings["first_buy"] = holdings["symbol"].map(lots.groupby("symbol")["date"].min())

    eurusd = cached_close("EURUSD=X", "max")
    eurusd.index = eurusd.index.tz_localize(None).normalize() if eurusd.index.tz is not None else eurusd.index.normalize()
    events, kept_by_market, unknown_markets = [], {}, set()
    tax_table = load_dividend_tax(os.path.dirname(ledger_path) or ".")
    for symbol, currency in zip(holdings["symbol"], holdings["currency"]):
        if symbol not in good:
            continue
        mine = lots[lots["symbol"] == symbol]
        kept, known = _kept_share(symbol, tax_table)
        kept_by_market[_market_key(symbol)] = kept
        if not known:
            unknown_markets.add(_market_key(symbol))
        for ex_date, per_share in _dividends(symbol).items():
            shares = mine.loc[mine["date"] < ex_date, "quantity"].sum()      # bought before the ex-date
            if shares <= 0 or per_share <= 0:
                continue
            if currency == "EUR":
                rate = 1.0
            elif currency == "USD":
                rate = float(eurusd.reindex(eurusd.index.union([ex_date])).ffill().loc[ex_date])
            else:
                raise ValueError(f"No exchange rate set up for {currency} ({symbol})")
            events.append(dict(symbol=symbol, date=ex_date, eur=shares * per_share * kept / rate))
    events = pd.DataFrame(events, columns=["symbol", "date", "eur"])
    holdings["dividends_eur"] = holdings["symbol"].map(events.groupby("symbol")["eur"].sum()).fillna(0.0)

    kept_text = ", ".join(f"{m} {k:.1%}" for m, k in sorted(kept_by_market.items()))
    notes.append(f"Dividends are estimated from Yahoo's payouts × the shares you held on each ex-date × the share you keep after the tax at source "
                 f"(`{DIVIDEND_TAX_FILE}`) and the Belgian {BELGIAN_TAX:.0%} (kept: {kept_text}). Total estimated: €{events['eur'].sum():,.0f}.")
    if unknown_markets:
        notes.append("**Market not in the tax table** (30% at source assumed): " + ", ".join(sorted(unknown_markets)) +
                     f". Add a row to `{DIVIDEND_TAX_FILE}`, or it is learned from the first dividend in a History PDF.")
    LEDGER.update(lots=lots, dividend_events=events, notes=notes, first_buy=lots["date"].min())
    return holdings

def xirr(flows):
    """Annualised return (%) of dated cash flows [(date, amount), ...]: money out negative, money in positive."""
    flows = [(pd.Timestamp(d), a) for d, a in flows]
    t0 = min(d for d, _ in flows)
    npv = lambda r: sum(a / (1 + r) ** ((d - t0).days / 365) for d, a in flows)
    lo, hi = -0.99, 100.0
    if npv(lo) * npv(hi) > 0:
        return float("nan")
    for _ in range(200):
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if npv(lo) * npv(mid) <= 0 else (mid, hi)
    return (lo + hi) / 2 * 100

def add_return_columns(holdings):
    """Total return (price + dividends) and annualised return per holding, from the real dates."""
    lots, events = LEDGER.get("lots"), LEDGER.get("dividend_events")
    holdings["total_return_pct"] = np.nan
    holdings["xirr_pct"] = np.nan
    if lots is None or lots.empty:
        return
    today = pd.Timestamp.today().normalize()
    holdings["total_return_pct"] = (holdings["market_value_eur"] + holdings["dividends_eur"]) / holdings["cost_basis_eur"] * 100 - 100
    for i, row in holdings.iterrows():
        mine, divs = lots[lots["symbol"] == row["symbol"]], events[events["symbol"] == row["symbol"]]
        if mine.empty:
            holdings.loc[i, "total_return_pct"] = np.nan
            continue
        flows = [(d, -c) for d, c in zip(mine["date"], mine["cost_eur"])] + list(zip(divs["date"], divs["eur"])) + [(today, row["market_value_eur"])]
        holdings.loc[i, "xirr_pct"] = xirr(flows)
    inside = holdings["symbol"].isin(lots["symbol"])
    flows = [(d, -c) for d, c in zip(lots["date"], lots["cost_eur"])] + list(zip(events["date"], events["eur"])) \
            + [(today, holdings.loc[inside, "market_value_eur"].sum())]
    LEDGER["portfolio_xirr"] = xirr(flows)
    LEDGER["portfolio_total_return"] = (holdings.loc[inside, "market_value_eur"].sum() + holdings.loc[inside, "dividends_eur"].sum()) \
                                       / holdings.loc[inside, "cost_basis_eur"].sum() * 100 - 100


# ---------- no Excel needed: holdings and cash from the transactions file alone ----------
def holdings_from_ledger(folder="."):
    """What you hold = buys minus sells in the ledger; cash = the last cash file minus what you spent since."""
    path = latest_ledger(folder)
    ledger = read_ledger(path)
    if "symbol" not in ledger or ledger["symbol"].isna().any():
        raise ValueError("The transactions file has no Yahoo symbols yet. Run import_history_pdfs() first.")
    rows = []
    for symbol, g in ledger.groupby("symbol"):
        shares = g.loc[g["type"] == "buy", "quantity"].sum() - g.loc[g["type"] == "sell", "quantity"].sum()
        if shares <= 1e-9:
            continue                                                     # fully sold
        if (g["type"] == "sell").any():
            raise ValueError(f"The ledger contains sales of {symbol}, which the cost calculation does not handle yet.")
        buys = g[g["type"] == "buy"]
        rows.append(dict(symbol=symbol, shares=shares, avg_cost=(buys["quantity"] * buys["price"]).sum() / buys["quantity"].sum(),
                         currency=buys["currency"].iloc[0], cost_basis_eur=(buys["amount"] / buys["fx"]).sum()))
    h = pd.DataFrame(rows)
    as_of = dt.date.today().isoformat()
    lines = ["symbol,shares,avg_cost,currency,cost_basis_eur,as_of"]
    for r in h.itertuples():
        shares = int(r.shares) if r.shares == int(r.shares) else r.shares
        lines.append(f"{r.symbol},{shares},{r.avg_cost:.2f},{r.currency},{r.cost_basis_eur:.2f},{as_of}")
    holdings_path = os.path.join(folder, f"holdings-{as_of}.csv")
    print(f"Built {len(h)} positions from {os.path.basename(path)} (no Excel).")
    display(h[["symbol", "shares", "currency", "avg_cost", "cost_basis_eur"]])
    _write_if_ok(holdings_path, "\n".join(lines) + "\n")

    cash_path = os.path.join(folder, f"cash-{as_of}.csv")
    previous = sorted(glob.glob(os.path.join(folder, "cash-*.csv")))
    if os.path.isfile(cash_path):
        print(f"  Cash: using {os.path.basename(cash_path)}.")
    elif previous:
        cash = pd.read_csv(previous[-1])
        since = pd.to_datetime(cash["as_of"].iloc[0])
        spent = ledger[ledger["date"] > since]
        delta = float((np.where(spent["type"] == "buy", -1, 1) * spent["amount"] / spent["fx"]).sum())
        if "EUR" not in set(cash["currency"]):
            cash = pd.concat([cash, pd.DataFrame([dict(currency="EUR", amount=0.0, as_of=cash["as_of"].iloc[0])])], ignore_index=True)
        cash.loc[cash["currency"] == "EUR", "amount"] += delta
        _write_if_ok(cash_path, "currency,amount,as_of\n" + "".join(f"{r.currency},{r.amount:.2f},{as_of}\n" for r in cash.itertuples()))
        print(f"  Cash is an estimate: {os.path.basename(previous[-1])} {delta:+,.2f} EUR of trades since {since:%d %b %Y}. Export the Excel to get the exact figure.")
    else:
        print("  No cash file found, so no cash slice.")
    return holdings_path

def build_holdings(folder="."):
    """Excel if it is at least as recent as your last transaction, otherwise the transactions file alone."""
    ledger_path = latest_ledger(folder)
    excel = find_bank_excel(required=False) if ledger_path else find_bank_excel()
    if excel and ledger_path:
        last_trade = read_ledger(ledger_path)["date"].max().date()
        if dt.date.fromtimestamp(os.path.getmtime(excel)) < last_trade:
            print(f"  The Excel ({os.path.basename(excel)}) is older than your last transaction ({last_trade}): building from the transactions file instead.")
            excel = None
    return excel_to_csv(excel, folder) if excel else holdings_from_ledger(folder)


## 3. Run

In [ ]:
# ===== RUN: reads new History PDFs from Downloads, then draws the dashboard =====
import_history_pdfs()                  # adds new buys/sells (and finds each stock's Yahoo symbol from its name)
show_dashboard(build_holdings())       # uses a recent portfolio Excel from Downloads if there is one, otherwise just your transactions

# Already have a holdings CSV? Comment out the two lines above and use this one instead:
# show_dashboard(ask_for_csv())
